# Installation

In [ ]:
# This notebook evaluates CDFM on the [Causal Chamber](https://causalchamber.ai/) a real-world physical testbed for causal discovery.
!pip install cdfm-base

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
Cloning into 'CDFM'...


In [16]:
import os, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cdfm import CDFM
from cdfm.utils import evaluate_graph, suggest_threshold, edge_auroc


# Paths
DATA_DIR = "../tests/data/causal_chamber"
DAG_PATH=  "../tests/data/causal_chamber/dag.csv"
A1_DIR = os.path.join(DATA_DIR, "task_a1")
A2_DIR = os.path.join(DATA_DIR, "task_a2")

# 20 core causal variables
DAG_VARS = [
    'red', 'green', 'blue', 'current',
    'pol_1', 'pol_2',
    'ir_1', 'vis_1', 'ir_2', 'vis_2', 'ir_3', 'vis_3',
    'angle_1', 'angle_2',
    'l_11', 'l_12', 'l_21', 'l_22', 'l_31', 'l_32',
]

# Task A2: 20 intervention datasets
TASK_A2 = [
    'uniform_red_strong', 'uniform_green_strong', 'uniform_blue_strong',
    'uniform_v_c_strong',
    'uniform_t_ir_1_strong', 'uniform_t_ir_2_strong', 'uniform_t_ir_3_strong',
    'uniform_t_vis_1_strong', 'uniform_t_vis_2_strong', 'uniform_t_vis_3_strong',
    'uniform_pol_1_strong', 'uniform_pol_2_strong',
    'uniform_v_angle_1_strong', 'uniform_v_angle_2_strong',
    'uniform_l_11_mid', 'uniform_l_12_mid',
    'uniform_l_21_mid', 'uniform_l_22_mid',
    'uniform_l_31_mid', 'uniform_l_32_mid',
]

print(f"Variables: {len(DAG_VARS)}  |  A2 datasets: {len(TASK_A2)}  |  DAG: {DAG_PATH}")


Variables: 20  |  A2 datasets: 20  |  DAG: ../tests/data/causal_chamber/dag.csv


# Load CDFM Model


In [6]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = CDFM.from_pretrained("DMIRLAB/CDFM")

print(f"{model.info['architecture']}  |  {model.info['parameters']:,} params")

Device: cpu


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


README.md: 0.00B [00:00, ?B/s]

d:\ProgramData\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\79372\.cache\huggingface\hub\models--DMIRLAB--CDFM. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/717 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/38.7M [00:00<?, ?B/s]

CDFM  |  9,662,124 params


# Learn Causal Structure


In [12]:
def load_data(path):
    """Load CSV, extract DAG_VARS columns."""
    df = pd.read_csv(path)
    vp = [v for v in DAG_VARS if v in df.columns]
    return df[vp].values.astype(np.float32)


def load_gt(dag_path=None, var_names=DAG_VARS):
    """Build (D,D) adjacency from dag.csv."""
    dag = pd.read_csv(dag_path)
    v2i = {v: i for i, v in enumerate(var_names)}
    D = len(var_names)
    adj = np.zeros((D, D), dtype=np.int32)
    for _, row in dag.iterrows():
        s, t = row['source'], row['target']
        if s in v2i and t in v2i:
            adj[v2i[s], v2i[t]] = 1
    return adj, int(adj.sum())


gt, n_true = load_gt(dag_path=DAG_PATH, var_names=DAG_VARS)
print(f"Ground truth: {n_true} edges, {len(DAG_VARS)} variables")


Ground truth: 39 edges, 20 variables


# Task A1


In [ ]:
print("=" * 60)
print("TASK A1")
print("=" * 60)

a1_data = load_data(os.path.join(A1_DIR, "uniform_reference.csv"))
print(f"Data: N={a1_data.shape[0]}, D={a1_data.shape[1]}")


a1_result = model.predict(a1_data)
a1_m = evaluate_graph(a1_result.adjacency, gt)
a1_auroc = edge_auroc(a1_result.logits, gt)
print(f"F1={a1_m['f1']:.4f}  Prec={a1_m['precision']:.4f}  Rec={a1_m['recall']:.4f}"
      f"TP={a1_m['tp']}  FP={a1_m['fp']}  FN={a1_m['fn']}  SHD={a1_m['shd']}",
      f"AUROC: {a1_auroc:.4f}")

TASK A1
Data: N=10000, D=20
F1=0.7273  Prec=0.8889  Rec=0.6154TP=24  FP=3  FN=15  SHD=17 AUROC: 0.9353


# Task A2

In [ ]:
print("\n" + "=" * 60)
print("TASK A2")
print("=" * 60)

all_adjs, all_ts, a2_logits = [], [], []
for ds in TASK_A2:
    r = model.predict(load_data(os.path.join(A2_DIR, f"{ds}.csv")))
    a2_logits.append(r.logits)
    # Auto-calibrate threshold per dataset
    t_i = suggest_threshold(r.logits, weights=model.calibration_coefficients_,
                            features=model.calibration_feature_names_)
    probs = 1 / (1 + np.exp(-np.clip(r.logits, -50, 50)))
    np.fill_diagonal(probs, 0)
    adj = (probs > t_i).astype(np.int32)
    np.fill_diagonal(adj, 0)
    all_adjs.append(adj)
    all_ts.append(t_i)

vote = np.stack(all_adjs).mean(axis=0)
a2_pred = (vote >= 5/20).astype(np.int32)
a2_m = evaluate_graph(a2_pred, gt)

print(f"F1={a2_m['f1']:.4f}  Prec={a2_m['precision']:.4f}  Rec={a2_m['recall']:.4f}  "
      f"TP={a2_m['tp']}  FP={a2_m['fp']}  FN={a2_m['fn']}  SHD={a2_m['shd']}")

# Average logits across all 20 datasets.
avg_logits = np.stack(a2_logits).mean(axis=0)
np.fill_diagonal(avg_logits, 0)
a2_auroc = edge_auroc(avg_logits, gt)
a2_m = evaluate_graph(avg_logits>0.7, gt)
print(f"F1={a2_m['f1']:.4f}  Prec={a2_m['precision']:.4f}  Rec={a2_m['recall']:.4f}  "
      f"TP={a2_m['tp']}  FP={a2_m['fp']}  FN={a2_m['fn']}  SHD={a2_m['shd']}",f"AUROC: {a2_auroc:.4f}")



TASK A2
F1=0.7353  Prec=0.8621  Rec=0.6410  TP=25  FP=4  FN=14  SHD=18
F1=0.7353  Prec=0.8621  Rec=0.6410  TP=25  FP=4  FN=14  SHD=18 AUROC: 0.9686
